In [1]:
import pandas as pd

In [49]:
return_average = pd.read_csv("0.003_total_del_avg.csv",  header=0)

In [50]:
return_average.rename(columns={"Unnamed: 0": "date"}, inplace=True)

In [52]:
return_average["SAC"].describe()

count    76.000000
mean      0.008383
std       0.043258
min      -0.134798
25%      -0.011921
50%       0.009868
75%       0.030783
max       0.121009
Name: SAC, dtype: float64

In [53]:
return_average["SAC"].skew()

np.float64(-0.7538194401058241)

In [54]:
desc = return_average["SAC"].describe()
std_error = desc["std"] / (desc["count"] ** 0.5)
print(std_error)

0.004962072589761025


In [55]:
return_average["SAC"].kurtosis()

np.float64(2.47752718279302)

In [56]:
return_average = return_average[["date", "SAC"]]

In [57]:
return_average

,date,SAC
0,2019-01-30,-0.014740
1,2019-02-28,0.019843
2,2019-03-28,-0.008855
3,2019-04-26,0.019544
4,2019-05-24,-0.027554
...,...,...
71,2024-09-20,0.003555
72,2024-10-18,0.013755
73,2024-11-15,-0.015770
74,2024-12-16,0.053909


In [58]:
return_average.rename(columns={"SAC": "return"}, inplace=True)

/tmp/ipykernel_1062457/1625460571.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return_average.rename(columns={"SAC": "return"}, inplace=True)


In [59]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac

def newey_west_tstat(returns, maxlags=1):
    """
    논문 방식에 따른 Newey-West t-통계량 계산 함수
    입력:
        returns: 수익률 벡터 (list, np.array, pd.Series)
        maxlags: Newey-West 보정에 사용할 최대 시차
    출력:
        (평균 수익률, NW 표준오차, NW t-통계량)
    """
    returns = np.asarray(returns)
    T = len(returns)
    X = np.ones((T, 1))  # 상수항만 포함 (평균 추정)
    
    model = sm.OLS(returns, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
    nw_cov = cov_hac(model, nlags=maxlags)
    # nw_se = np.sqrt(nw_cov[0, 0])
    # t_stat = model.params[0] / nw_se
    
    return model.params[0], model.bse[0], model.tvalues[0]


In [60]:

# 계산 실행
mean_return, nw_se, nw_tstat = newey_west_tstat(return_average["return"], maxlags=12)


In [61]:
from scipy.stats import t



# 단측 검정 (우측): P(T > t)
p_value = 1 - t.cdf(nw_tstat, df=len(return_average))

print(f"p-value = {p_value:.6f}")


p-value = 0.068668


In [62]:
nw_tstat

np.float64(1.5016289285442335)

In [31]:
return_average

,date,return
0,2019-01-30,0.014351
1,2019-02-28,0.017790
2,2019-03-28,-0.006875
3,2019-04-26,0.022650
4,2019-05-24,-0.025741
...,...,...
71,2024-09-20,0.025878
72,2024-10-18,0.000756
73,2024-11-15,-0.023999
74,2024-12-16,0.062063


In [11]:
np.quantile(return_average['return'], 0.05)

np.float64(-0.04370586275)